# Cisco Netflow Data Generation

This notebook demonstrates how to generate synthetic Cisco Netflow records using Rockfish's Entity Data Generator.

**What this notebook shows:**
- Creating a Rockfish `DataSchema` for Netflow records
- Generating realistic protocol-port combinations (TCP/UDP with appropriate ports)
- Creating IPv4 addresses in proper format
- Modeling realistic traffic patterns with bytes, packets, and flow durations

**Netflow Fields Generated:**
- Source/Destination IP addresses (IPv4)
- Source/Destination ports (realistic ranges based on protocol)
- Protocol (TCP, UDP, ICMP)
- Bytes, packets, and flow duration
- TCP flags (for TCP flows)
- Interface information

## Setup and Imports

In [1]:
import rockfish as rf
import rockfish.actions as ra
from rockfish.actions.ent import (
    CategoricalParams,
    Column,
    ColumnCategoryType,
    ColumnType,
    DataSchema,
    Derivation,
    DerivationFunctionType,
    Domain,
    DomainType,
    Entity,
    EntityRelationship,
    EntityRelationshipType,
    GlobalTimestamp,
    IDParams,
    MapValuesParams,
    NormalDistParams,
    SampleFromColumnParams,
    SequentialIntParams,
    UniformDistParams,
    ExponentialDistParams,
)
from dotenv import load_dotenv
import pandas as pd

In [2]:
# Connect to the Rockfish platform using your API Key
load_dotenv()
conn = rf.Connection.from_env()

## Create Schema

We'll generate Netflow data with the following design:

### Entities:
1. **network_host**: Internal network hosts with IPv4 addresses
2. **external_host**: External hosts (internet destinations)
3. **service**: Common network services with their protocols and ports
4. **netflow_record**: Individual flow records linking hosts and services

### Realistic Patterns:
- **Protocol-Port Mapping**: Services define protocol (TCP/UDP) and well-known ports
- **Port Ranges**: 
  - Well-known ports: 0-1023 (HTTP, HTTPS, DNS, SSH, etc.)
  - Registered ports: 1024-49151 (common applications)
  - Dynamic/Ephemeral ports: 49152-65535 (client-side source ports)
- **IPv4 Format**: Addresses generated as octets then combined

In [3]:
# Configuration parameters
N_INTERNAL_HOSTS = 50      # Internal network hosts
N_EXTERNAL_HOSTS = 100     # External destinations
N_SERVICES = 15            # Network services (protocol + port combinations)
N_NETFLOW_RECORDS = 5000   # Number of flow records to generate

In [6]:
def create_netflow_schema(
    n_internal_hosts=50,
    n_external_hosts=100,
    n_services=15,
    n_netflow_records=5000,
) -> DataSchema:
    """Create the Cisco Netflow data schema."""
    
    # ENTITY 1: network_host (internal hosts)
    # Internal hosts on the local network (e.g., 10.x.x.x or 192.168.x.x)
    network_host = Entity(
        name="network_host",
        cardinality=n_internal_hosts,
        columns=[
            # Host ID
            Column(
                name="host_id",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.ID,
                    params=IDParams(template_str="HOST_{id}"),
                ),
            ),
            # IPv4 octet 1 (private network: 10.x.x.x)
            Column(
                name="ip_octet_1",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=[10],  # Class A private network
                        with_replacement=True,
                    ),
                ),
            ),
            # IPv4 octet 2
            Column(
                name="ip_octet_2",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=[1, 2, 3, 4, 5],  # Subnets
                        with_replacement=True,
                    ),
                ),
            ),
            # IPv4 octet 3
            Column(
                name="ip_octet_3",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.UNIFORM_DIST,
                    params=UniformDistParams(lower=0, upper=255),
                ),
            ),
            # IPv4 octet 4
            Column(
                name="ip_octet_4",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.UNIFORM_DIST,
                    params=UniformDistParams(lower=1, upper=254),  # Avoid .0 and .255
                ),
            ),
            # Host type
            Column(
                name="host_type",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=["workstation", "workstation", "workstation", "server", "printer", "iot_device"],
                        with_replacement=True,
                    ),
                ),
            ),
        ],
    )
    
    # ENTITY 2: external_host (internet destinations)
    external_host = Entity(
        name="external_host",
        cardinality=n_external_hosts,
        columns=[
            # External host ID
            Column(
                name="ext_host_id",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.ID,
                    params=IDParams(template_str="EXT_{id}"),
                ),
            ),
            # IPv4 octet 1 (public ranges, avoiding reserved)
            Column(
                name="ext_ip_octet_1",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        # Common public IP first octets (avoiding private/reserved)
                        values=[8, 13, 17, 20, 23, 31, 34, 35, 40, 52, 54, 64, 72, 74, 93, 104, 142, 151, 157, 172, 199, 204, 208, 216],
                        with_replacement=True,
                    ),
                ),
            ),
            # IPv4 octet 2
            Column(
                name="ext_ip_octet_2",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.UNIFORM_DIST,
                    params=UniformDistParams(lower=0, upper=255),
                ),
            ),
            # IPv4 octet 3
            Column(
                name="ext_ip_octet_3",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.UNIFORM_DIST,
                    params=UniformDistParams(lower=0, upper=255),
                ),
            ),
            # IPv4 octet 4
            Column(
                name="ext_ip_octet_4",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.UNIFORM_DIST,
                    params=UniformDistParams(lower=1, upper=254),
                ),
            ),
            # Category of external host
            Column(
                name="ext_category",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=["cdn", "cloud_service", "web_server", "dns_server", "mail_server", "api_endpoint"],
                        with_replacement=True,
                    ),
                ),
            ),
        ],
    )
    
    # ENTITY 3: service (protocol + port definitions)
    # Define common services with realistic protocol/port combinations
    service = Entity(
        name="service",
        cardinality=n_services,
        columns=[
            # Service ID
            Column(
                name="service_id",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.ID,
                    params=IDParams(template_str="SVC_{id}"),
                ),
            ),
            # Service name - realistic protocol/port combinations
            Column(
                name="service_name",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=[
                            "HTTP",      # TCP/80
                            "HTTPS",     # TCP/443
                            "SSH",       # TCP/22
                            "DNS",       # UDP/53 (also TCP)
                            "SMTP",      # TCP/25
                            "IMAP",      # TCP/143
                            "IMAPS",     # TCP/993
                            "POP3",      # TCP/110
                            "FTP",       # TCP/21
                            "MYSQL",     # TCP/3306
                            "POSTGRESQL",# TCP/5432
                            "REDIS",     # TCP/6379
                            "NTP",       # UDP/123
                            "SNMP",      # UDP/161
                            "RDP",       # TCP/3389
                        ],
                        with_replacement=False,
                    ),
                ),
            ),
            # Protocol number (6=TCP, 17=UDP, 1=ICMP)
            Column(
                name="protocol",
                data_type="int64",
                column_type=ColumnType.DERIVED,
                column_category_type=ColumnCategoryType.METADATA,
                derivation=Derivation(
                    function_type=DerivationFunctionType.MAP_VALUES,
                    dependent_columns=["service_name"],
                    params=MapValuesParams(
                        mapping=[
                            {"from": "HTTP", "to": "6"},
                            {"from": "HTTPS", "to": "6"},
                            {"from": "SSH", "to": "6"},
                            {"from": "DNS", "to": "17"},      # Primary DNS is UDP
                            {"from": "SMTP", "to": "6"},
                            {"from": "IMAP", "to": "6"},
                            {"from": "IMAPS", "to": "6"},
                            {"from": "POP3", "to": "6"},
                            {"from": "FTP", "to": "6"},
                            {"from": "MYSQL", "to": "6"},
                            {"from": "POSTGRESQL", "to": "6"},
                            {"from": "REDIS", "to": "6"},
                            {"from": "NTP", "to": "17"},
                            {"from": "SNMP", "to": "17"},
                            {"from": "RDP", "to": "6"},
                        ],
                        default="6",
                    ),
                ),
            ),
            # Destination port (well-known port for the service)
            Column(
                name="dst_port",
                data_type="int64",
                column_type=ColumnType.DERIVED,
                column_category_type=ColumnCategoryType.METADATA,
                derivation=Derivation(
                    function_type=DerivationFunctionType.MAP_VALUES,
                    dependent_columns=["service_name"],
                    params=MapValuesParams(
                        mapping=[
                            {"from": "HTTP", "to": "80"},
                            {"from": "HTTPS", "to": "443"},
                            {"from": "SSH", "to": "22"},
                            {"from": "DNS", "to": "53"},
                            {"from": "SMTP", "to": "25"},
                            {"from": "IMAP", "to": "143"},
                            {"from": "IMAPS", "to": "993"},
                            {"from": "POP3", "to": "110"},
                            {"from": "FTP", "to": "21"},
                            {"from": "MYSQL", "to": "3306"},
                            {"from": "POSTGRESQL", "to": "5432"},
                            {"from": "REDIS", "to": "6379"},
                            {"from": "NTP", "to": "123"},
                            {"from": "SNMP", "to": "161"},
                            {"from": "RDP", "to": "3389"},
                        ],
                        default="80",
                    ),
                ),
            ),
        ],
    )
    
    # ENTITY 4: netflow_record
    # Individual flow records with traffic measurements
    netflow_record = Entity(
        name="netflow_record",
        cardinality=n_netflow_records,
        columns=[
            # Flow ID
            Column(
                name="flow_id",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.ID,
                    params=IDParams(template_str="FLOW_{id}"),
                ),
            ),
            # Foreign key to source host (internal)
            Column(
                name="fk_src_host_id",
                data_type="string",
                column_type=ColumnType.FOREIGN_KEY,
                column_category_type=ColumnCategoryType.METADATA,
            ),
            # Foreign key to destination host (external)
            Column(
                name="fk_dst_host_id",
                data_type="string",
                column_type=ColumnType.FOREIGN_KEY,
                column_category_type=ColumnCategoryType.METADATA,
            ),
            # Foreign key to service
            Column(
                name="fk_service_id",
                data_type="string",
                column_type=ColumnType.FOREIGN_KEY,
                column_category_type=ColumnCategoryType.METADATA,
            ),
            # Source port (ephemeral port range: 49152-65535)
            Column(
                name="src_port",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.UNIFORM_DIST,
                    params=UniformDistParams(lower=49152, upper=65535),
                ),
            ),
            # Bytes transferred (exponential distribution - most flows are small)
            Column(
                name="bytes",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.EXPONENTIAL_DIST,
                    params=ExponentialDistParams(scale=50000),  # Mean ~50KB
                ),
            ),
            # Packets count
            Column(
                name="packets",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.EXPONENTIAL_DIST,
                    params=ExponentialDistParams(scale=50),  # Mean ~50 packets
                ),
            ),
            # Flow duration in milliseconds
            Column(
                name="duration_ms",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.EXPONENTIAL_DIST,
                    params=ExponentialDistParams(scale=30000),  # Mean ~30 seconds
                ),
            ),
            # TCP flags (realistic combinations)
            Column(
                name="tcp_flags",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        # Common TCP flag combinations
                        values=[
                            "SYN",           # Connection initiation
                            "SYN-ACK",       # Connection response
                            "ACK",           # Most common (data transfer)
                            "ACK",           # Weighted for frequency
                            "ACK",           # Weighted for frequency
                            "PSH-ACK",       # Data push
                            "PSH-ACK",       # Weighted
                            "FIN-ACK",       # Connection termination
                            "RST",           # Connection reset
                            "RST-ACK",       # Reset acknowledgment
                        ],
                        with_replacement=True,
                    ),
                ),
            ),
            # Input interface
            Column(
                name="input_interface",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=["Gi0/0", "Gi0/1", "Gi0/2", "Gi0/3", "Gi1/0", "Gi1/1"],
                        with_replacement=True,
                    ),
                ),
            ),
            # Output interface
            Column(
                name="output_interface",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=["Gi0/0", "Gi0/1", "Gi0/2", "Gi0/3", "Gi1/0", "Gi1/1"],
                        with_replacement=True,
                    ),
                ),
            ),
            # Type of Service (ToS) / DSCP
            Column(
                name="tos",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        # Common ToS values: 0 (best effort), 32 (CS1), 40 (CS5), 46 (EF)
                        values=[0, 0, 0, 0, 0, 32, 40, 46],  # Mostly best effort
                        with_replacement=True,
                    ),
                ),
            ),
            # Flow direction
            Column(
                name="direction",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=["outbound", "outbound", "outbound", "inbound"],  # More outbound traffic
                        with_replacement=True,
                    ),
                ),
            ),
        ],
    )
    
    # ENTITY RELATIONSHIPS
    relationships = [
        # network_host -> netflow_record (one-to-many)
        # An internal host can have many flows
        EntityRelationship(
            parent_entity="network_host",
            child_entity="netflow_record",
            relationship_type=EntityRelationshipType.ONE_TO_MANY,
            join_columns={
                "host_id": "fk_src_host_id",
            },
        ),
        # external_host -> netflow_record (one-to-many)
        # An external host can be destination for many flows
        EntityRelationship(
            parent_entity="external_host",
            child_entity="netflow_record",
            relationship_type=EntityRelationshipType.ONE_TO_MANY,
            join_columns={
                "ext_host_id": "fk_dst_host_id",
            },
        ),
        # service -> netflow_record (one-to-many)
        # A service can be used by many flows
        EntityRelationship(
            parent_entity="service",
            child_entity="netflow_record",
            relationship_type=EntityRelationshipType.ONE_TO_MANY,
            join_columns={
                "service_id": "fk_service_id",
            },
        ),
    ]
    
    # Global timestamp for time-series flows
    global_ts = GlobalTimestamp(
        t_start="2025-01-15T00:00:00+00:00",
        t_end="2025-01-15T23:59:59+00:00",
        time_interval="1min",
    )
    
    return DataSchema(
        entities=[network_host, external_host, service, netflow_record],
        entity_relationships=relationships,
        global_timestamp=global_ts,
    )

In [7]:
# Create the schema instance
netflow_schema = create_netflow_schema(
    n_internal_hosts=N_INTERNAL_HOSTS,
    n_external_hosts=N_EXTERNAL_HOSTS,
    n_services=N_SERVICES,
    n_netflow_records=N_NETFLOW_RECORDS,
)

## Run Data Generation

We'll use a **Rockfish Workflow** to run a data generation job on the Rockfish platform.

In [8]:
config = ra.GenerateFromDataSchema.Config(
    schema=netflow_schema,
    upload_datasets=True,
)
generate = ra.GenerateFromDataSchema(config)

In [9]:
builder = rf.WorkflowBuilder()
builder.add(generate)
workflow = await builder.start(conn)
print(f"Workflow ID: {workflow.id()}")

Workflow ID: 3KHwkph9shZ2ldQ0hmNYcK


In [10]:
async for log in workflow.logs(level=rf.events.LogLevel.DEBUG):
    print(log)

2026-02-15T22:36:03.456509Z generate-from-data-schema: INFO Generating 4 entities: network_host, external_host, service, netflow_record
2026-02-15T22:36:03.470857Z generate-from-data-schema: INFO Starting data generation...
2026-02-15T22:36:03.507416Z generate-from-data-schema: INFO Generated 4 entity tables
2026-02-15T22:36:03.518824Z generate-from-data-schema: INFO Creating dataset for entity 'network_host': 50 rows
2026-02-15T22:36:03.742662Z generate-from-data-schema: INFO Uploaded dataset 'network_host' (58JdiprgcAjHJjeBBsveTf): 50 rows
2026-02-15T22:36:03.765757Z generate-from-data-schema: INFO Creating dataset for entity 'external_host': 100 rows
2026-02-15T22:36:03.931988Z generate-from-data-schema: INFO Uploaded dataset 'external_host' (4hplIIp5zI32xOlISvb0Ag): 100 rows
2026-02-15T22:36:03.956106Z generate-from-data-schema: INFO Creating dataset for entity 'service': 15 rows
2026-02-15T22:36:04.107390Z generate-from-data-schema: INFO Uploaded dataset 'service' (24NJYsi82WMRkG6

## Retrieve Generated Datasets

We'll retrieve datasets for all entities: `network_host`, `external_host`, `service`, and `netflow_record`.

In [11]:
datasets = await workflow.datasets().collect()
print(f"Generated {len(datasets)} datasets")

network_host_dataset = None
external_host_dataset = None
service_dataset = None
netflow_dataset = None

for remote_ds in datasets:
    ds = await remote_ds.to_local(conn)
    if ds.name() == "network_host":
        network_host_dataset = ds
    elif ds.name() == "external_host":
        external_host_dataset = ds
    elif ds.name() == "service":
        service_dataset = ds
    elif ds.name() == "netflow_record":
        netflow_dataset = ds

Generated 4 datasets


## Explore Network Hosts

Internal hosts with IPv4 addresses in the 10.x.x.x private range.

In [12]:
network_host_df = network_host_dataset.to_pandas()
print(f"Network Host dataset: {network_host_dataset.table.num_rows} rows")

# Construct full IPv4 addresses
network_host_df['src_ip'] = (
    network_host_df['ip_octet_1'].astype(str) + '.' +
    network_host_df['ip_octet_2'].astype(str) + '.' +
    network_host_df['ip_octet_3'].astype(str) + '.' +
    network_host_df['ip_octet_4'].astype(str)
)

network_host_df[['host_id', 'src_ip', 'host_type']].head(10)

Network Host dataset: 50 rows


,host_id,src_ip,host_type
0,HOST_0,10.3.145.113,workstation
1,HOST_1,10.1.128.143,printer
2,HOST_2,10.3.22.19,workstation
3,HOST_3,10.5.60.205,server
4,HOST_4,10.1.9.139,iot_device
5,HOST_5,10.4.42.99,printer
6,HOST_6,10.5.107.60,workstation
7,HOST_7,10.3.247.248,iot_device
8,HOST_8,10.5.8.222,server
9,HOST_9,10.3.21.46,printer


In [13]:
print("Internal Hosts by Type:")
print(network_host_df["host_type"].value_counts())
print(f"\nSample IPv4 addresses:")
print(network_host_df['src_ip'].head(10).tolist())

Internal Hosts by Type:
host_type
workstation    26
printer        10
iot_device      8
server          6
Name: count, dtype: int64

Sample IPv4 addresses:
['10.3.145.113', '10.1.128.143', '10.3.22.19', '10.5.60.205', '10.1.9.139', '10.4.42.99', '10.5.107.60', '10.3.247.248', '10.5.8.222', '10.3.21.46']


## Explore External Hosts

External (internet) hosts with public IPv4 addresses.

In [14]:
external_host_df = external_host_dataset.to_pandas()
print(f"External Host dataset: {external_host_dataset.table.num_rows} rows")

# Construct full IPv4 addresses
external_host_df['dst_ip'] = (
    external_host_df['ext_ip_octet_1'].astype(str) + '.' +
    external_host_df['ext_ip_octet_2'].astype(str) + '.' +
    external_host_df['ext_ip_octet_3'].astype(str) + '.' +
    external_host_df['ext_ip_octet_4'].astype(str)
)

external_host_df[['ext_host_id', 'dst_ip', 'ext_category']].head(10)

External Host dataset: 100 rows


,ext_host_id,dst_ip,ext_category
0,EXT_0,8.229.200.38,web_server
1,EXT_1,64.98.113.127,web_server
2,EXT_2,208.34.126.88,mail_server
3,EXT_3,34.28.93.71,dns_server
4,EXT_4,35.192.43.170,mail_server
5,EXT_5,52.43.64.40,web_server
6,EXT_6,35.96.35.121,api_endpoint
7,EXT_7,17.234.207.74,cdn
8,EXT_8,35.185.155.127,dns_server
9,EXT_9,54.207.223.199,cdn


In [15]:
print("External Hosts by Category:")
print(external_host_df["ext_category"].value_counts())
print(f"\nSample Public IPv4 addresses:")
print(external_host_df['dst_ip'].head(10).tolist())

External Hosts by Category:
ext_category
web_server       19
mail_server      19
cloud_service    18
api_endpoint     17
dns_server       14
cdn              13
Name: count, dtype: int64

Sample Public IPv4 addresses:
['8.229.200.38', '64.98.113.127', '208.34.126.88', '34.28.93.71', '35.192.43.170', '52.43.64.40', '35.96.35.121', '17.234.207.74', '35.185.155.127', '54.207.223.199']


## Explore Services

Network services with their protocol numbers and destination ports.

In [16]:
service_df = service_dataset.to_pandas()
print(f"Service dataset: {service_dataset.table.num_rows} rows")

# Map protocol numbers to names
protocol_map = {6: 'TCP', 17: 'UDP', 1: 'ICMP'}
service_df['protocol_name'] = service_df['protocol'].map(protocol_map)

service_df[['service_id', 'service_name', 'protocol', 'protocol_name', 'dst_port']]

Service dataset: 15 rows


,service_id,service_name,protocol,protocol_name,dst_port
0,SVC_0,SSH,6,TCP,22
1,SVC_1,POSTGRESQL,6,TCP,5432
2,SVC_2,IMAP,6,TCP,143
3,SVC_3,HTTPS,6,TCP,443
4,SVC_4,SMTP,6,TCP,25
5,SVC_5,DNS,17,UDP,53
6,SVC_6,IMAPS,6,TCP,993
7,SVC_7,MYSQL,6,TCP,3306
8,SVC_8,NTP,17,UDP,123
9,SVC_9,REDIS,6,TCP,6379


In [17]:
print("Services by Protocol:")
print(service_df['protocol_name'].value_counts())
print("\nPort Range Analysis:")
print(f"  Min destination port: {service_df['dst_port'].min()}")
print(f"  Max destination port: {service_df['dst_port'].max()}")
print(f"  Well-known ports (< 1024): {(service_df['dst_port'] < 1024).sum()}")
print(f"  Registered ports (1024-49151): {((service_df['dst_port'] >= 1024) & (service_df['dst_port'] < 49152)).sum()}")

Services by Protocol:
protocol_name
TCP    12
UDP     3
Name: count, dtype: int64

Port Range Analysis:
  Min destination port: 21
  Max destination port: 6379
  Well-known ports (< 1024): 11
  Registered ports (1024-49151): 4


## Explore Netflow Records

Individual flow records with traffic measurements.

In [18]:
netflow_df = netflow_dataset.to_pandas()
print(f"Netflow Record dataset: {netflow_dataset.table.num_rows} rows")
netflow_df.head(10)

Netflow Record dataset: 5000 rows


,flow_id,fk_src_host_id,fk_dst_host_id,fk_service_id,src_port,bytes,packets,duration_ms,tcp_flags,input_interface,output_interface,tos,direction
0,FLOW_0,HOST_42,EXT_85,SVC_12,57776,31894,99,23281,FIN-ACK,Gi0/1,Gi0/3,0,outbound
1,FLOW_1,HOST_31,EXT_63,SVC_9,53103,41067,80,11083,ACK,Gi1/1,Gi1/0,0,inbound
2,FLOW_2,HOST_25,EXT_51,SVC_7,52888,3885,14,107289,ACK,Gi1/0,Gi1/1,0,outbound
3,FLOW_3,HOST_13,EXT_26,SVC_4,50989,116757,29,25187,RST-ACK,Gi0/2,Gi0/1,32,inbound
4,FLOW_4,HOST_15,EXT_30,SVC_4,52133,17068,4,66965,ACK,Gi0/2,Gi1/1,46,inbound
5,FLOW_5,HOST_2,EXT_4,SVC_0,59474,42861,54,42327,RST,Gi1/0,Gi0/2,0,outbound
6,FLOW_6,HOST_3,EXT_7,SVC_1,59825,4494,6,35471,ACK,Gi1/0,Gi0/0,0,inbound
7,FLOW_7,HOST_0,EXT_1,SVC_0,52314,42547,22,138732,RST-ACK,Gi0/0,Gi0/0,32,outbound
8,FLOW_8,HOST_8,EXT_17,SVC_2,54052,35950,97,26507,ACK,Gi1/0,Gi1/0,40,outbound
9,FLOW_9,HOST_40,EXT_81,SVC_12,59495,52166,17,46174,ACK,Gi1/1,Gi0/0,0,inbound


In [19]:
print("Netflow Statistics:")
print(f"\nSource Port Range (ephemeral):")
print(f"  Min: {netflow_df['src_port'].min()}")
print(f"  Max: {netflow_df['src_port'].max()}")
print(f"  Mean: {netflow_df['src_port'].mean():.0f}")

print(f"\nBytes per Flow:")
print(f"  Min: {netflow_df['bytes'].min()}")
print(f"  Max: {netflow_df['bytes'].max()}")
print(f"  Mean: {netflow_df['bytes'].mean():.0f}")
print(f"  Median: {netflow_df['bytes'].median():.0f}")

print(f"\nPackets per Flow:")
print(f"  Min: {netflow_df['packets'].min()}")
print(f"  Max: {netflow_df['packets'].max()}")
print(f"  Mean: {netflow_df['packets'].mean():.0f}")

print(f"\nTCP Flags Distribution:")
print(netflow_df['tcp_flags'].value_counts())

Netflow Statistics:

Source Port Range (ephemeral):
  Min: 49152
  Max: 65533
  Mean: 57389

Bytes per Flow:
  Min: 20
  Max: 582658
  Mean: 49917
  Median: 35274

Packets per Flow:
  Min: 0
  Max: 362
  Mean: 48

TCP Flags Distribution:
tcp_flags
ACK        1550
PSH-ACK     974
SYN         533
RST         530
FIN-ACK     498
SYN-ACK     461
RST-ACK     454
Name: count, dtype: int64


## Create Complete Netflow View

Join all entities to create a complete Netflow record with IP addresses and service details.

In [20]:
# Join netflow with network_host to get source IP
netflow_full = netflow_df.merge(
    network_host_df[['host_id', 'src_ip', 'host_type']],
    left_on='fk_src_host_id',
    right_on='host_id',
    how='left'
)

# Join with external_host to get destination IP
netflow_full = netflow_full.merge(
    external_host_df[['ext_host_id', 'dst_ip', 'ext_category']],
    left_on='fk_dst_host_id',
    right_on='ext_host_id',
    how='left'
)

# Join with service to get protocol and destination port
netflow_full = netflow_full.merge(
    service_df[['service_id', 'service_name', 'protocol', 'dst_port']],
    left_on='fk_service_id',
    right_on='service_id',
    how='left'
)

# Select and rename columns for standard Netflow format
netflow_export = netflow_full[[
    'flow_id',
    'src_ip',
    'dst_ip',
    'src_port',
    'dst_port',
    'protocol',
    'service_name',
    'bytes',
    'packets',
    'duration_ms',
    'tcp_flags',
    'input_interface',
    'output_interface',
    'tos',
    'direction',
    'host_type',
    'ext_category',
]].copy()

print(f"Complete Netflow dataset: {len(netflow_export)} rows")
netflow_export.head(15)

Complete Netflow dataset: 5000 rows


,flow_id,src_ip,dst_ip,src_port,dst_port,protocol,service_name,bytes,packets,duration_ms,tcp_flags,input_interface,output_interface,tos,direction,host_type,ext_category
0,FLOW_0,10.4.2.244,17.30.254.148,57776,3389,6,RDP,31894,99,23281,FIN-ACK,Gi0/1,Gi0/3,0,outbound,printer,api_endpoint
1,FLOW_1,10.1.155.52,172.74.48.68,53103,6379,6,REDIS,41067,80,11083,ACK,Gi1/1,Gi1/0,0,inbound,server,api_endpoint
2,FLOW_2,10.4.22.189,8.97.164.61,52888,3306,6,MYSQL,3885,14,107289,ACK,Gi1/0,Gi1/1,0,outbound,workstation,web_server
3,FLOW_3,10.1.20.88,17.34.147.91,50989,25,6,SMTP,116757,29,25187,RST-ACK,Gi0/2,Gi0/1,32,inbound,workstation,cdn
4,FLOW_4,10.1.19.148,74.176.191.215,52133,25,6,SMTP,17068,4,66965,ACK,Gi0/2,Gi1/1,46,inbound,iot_device,dns_server
5,FLOW_5,10.3.22.19,35.192.43.170,59474,22,6,SSH,42861,54,42327,RST,Gi1/0,Gi0/2,0,outbound,workstation,mail_server
6,FLOW_6,10.5.60.205,17.234.207.74,59825,5432,6,POSTGRESQL,4494,6,35471,ACK,Gi1/0,Gi0/0,0,inbound,server,cdn
7,FLOW_7,10.3.145.113,64.98.113.127,52314,22,6,SSH,42547,22,138732,RST-ACK,Gi0/0,Gi0/0,32,outbound,workstation,web_server
8,FLOW_8,10.5.8.222,72.88.246.35,54052,143,6,IMAP,35950,97,26507,ACK,Gi1/0,Gi1/0,40,outbound,server,web_server
9,FLOW_9,10.3.119.37,17.181.211.241,59495,3389,6,RDP,52166,17,46174,ACK,Gi1/1,Gi0/0,0,inbound,workstation,mail_server


## Validate Protocol-Port Patterns

Verify that protocol and port combinations are realistic.

In [21]:
# Verify protocol-port consistency
print("Protocol-Port Combinations:")
protocol_port_combos = netflow_export.groupby(['protocol', 'service_name', 'dst_port']).size().reset_index(name='count')
protocol_port_combos['protocol_name'] = protocol_port_combos['protocol'].map({6: 'TCP', 17: 'UDP', 1: 'ICMP'})
print(protocol_port_combos[['protocol_name', 'service_name', 'dst_port', 'count']].sort_values('count', ascending=False))

Protocol-Port Combinations:
   protocol_name service_name  dst_port  count
0            TCP          FTP        21    358
8            TCP          RDP      3389    354
11           TCP          SSH        22    347
12           UDP          DNS        53    345
3            TCP         IMAP       143    340
13           UDP          NTP       123    340
4            TCP        IMAPS       993    338
10           TCP         SMTP        25    338
7            TCP   POSTGRESQL      5432    337
6            TCP         POP3       110    336
5            TCP        MYSQL      3306    327
9            TCP        REDIS      6379    325
1            TCP         HTTP        80    316
14           UDP         SNMP       161    310
2            TCP        HTTPS       443    289


In [22]:
# Validate port ranges
print("\nPort Range Validation:")
print(f"Source ports in ephemeral range (49152-65535): {((netflow_export['src_port'] >= 49152) & (netflow_export['src_port'] <= 65535)).all()}")
print(f"Destination ports are well-known or registered: {(netflow_export['dst_port'] < 49152).all()}")

# Verify IP address formats
print(f"\nIPv4 Address Validation:")
print(f"All source IPs start with 10.x.x.x (private): {netflow_export['src_ip'].str.startswith('10.').all()}")
print(f"Sample source IPs: {netflow_export['src_ip'].head(5).tolist()}")
print(f"Sample destination IPs: {netflow_export['dst_ip'].head(5).tolist()}")


Port Range Validation:
Source ports in ephemeral range (49152-65535): True
Destination ports are well-known or registered: True

IPv4 Address Validation:
All source IPs start with 10.x.x.x (private): True
Sample source IPs: ['10.4.2.244', '10.1.155.52', '10.4.22.189', '10.1.20.88', '10.1.19.148']
Sample destination IPs: ['17.30.254.148', '172.74.48.68', '8.97.164.61', '17.34.147.91', '74.176.191.215']


## Traffic Analysis

In [23]:
# Traffic by service
print("Traffic by Service:")
traffic_by_service = netflow_export.groupby('service_name').agg({
    'bytes': 'sum',
    'packets': 'sum',
    'flow_id': 'count'
}).rename(columns={'flow_id': 'flow_count'}).sort_values('bytes', ascending=False)
traffic_by_service['bytes_gb'] = traffic_by_service['bytes'] / (1024**3)
print(traffic_by_service[['flow_count', 'packets', 'bytes_gb']].round(2))

Traffic by Service:
              flow_count  packets  bytes_gb
service_name                               
SMTP                 338    16425      0.02
DNS                  345    18244      0.02
RDP                  354    16936      0.02
IMAPS                338    16864      0.02
FTP                  358    17766      0.02
MYSQL                327    15221      0.02
NTP                  340    16449      0.02
POSTGRESQL           337    16750      0.02
REDIS                325    16589      0.02
IMAP                 340    14641      0.02
POP3                 336    17191      0.01
SNMP                 310    13485      0.01
SSH                  347    17220      0.01
HTTP                 316    14798      0.01
HTTPS                289    13326      0.01


In [24]:
# Traffic by internal host type
print("\nTraffic by Host Type:")
traffic_by_host = netflow_export.groupby('host_type').agg({
    'bytes': 'sum',
    'packets': 'sum',
    'flow_id': 'count'
}).rename(columns={'flow_id': 'flow_count'}).sort_values('bytes', ascending=False)
print(traffic_by_host)


Traffic by Host Type:
                 bytes  packets  flow_count
host_type                                  
workstation  129468810   125517        2575
printer       50216721    50858        1030
iot_device    39925484    36574         777
server        29974206    28956         618


In [25]:
# Top talkers (source IPs by bytes)
print("\nTop 10 Source IPs by Traffic Volume:")
top_sources = netflow_export.groupby('src_ip').agg({
    'bytes': 'sum',
    'flow_id': 'count'
}).rename(columns={'flow_id': 'flow_count'}).sort_values('bytes', ascending=False).head(10)
top_sources['bytes_mb'] = top_sources['bytes'] / (1024**2)
print(top_sources[['flow_count', 'bytes_mb']].round(2))


Top 10 Source IPs by Traffic Volume:
              flow_count  bytes_mb
src_ip                            
10.2.178.54          111      5.98
10.3.173.173         125      5.95
10.2.221.156         109      5.76
10.3.220.174         108      5.58
10.1.20.88           111      5.54
10.4.229.250         106      5.46
10.3.208.122         101      5.42
10.3.119.37           98      5.34
10.4.2.244           115      5.28
10.1.173.140         107      5.21


## Save Data to CSV

In [26]:
# Save all datasets to file
network_host_df.to_csv("network_hosts.csv", index=False)
external_host_df.to_csv("external_hosts.csv", index=False)
service_df.to_csv("services.csv", index=False)
netflow_export.to_csv("netflow_records.csv", index=False)

print("Data saved to CSV files:")
print("  - network_hosts.csv")
print("  - external_hosts.csv")
print("  - services.csv")
print("  - netflow_records.csv (complete joined view)")

Data saved to CSV files:
  - network_hosts.csv
  - external_hosts.csv
  - services.csv
  - netflow_records.csv (complete joined view)


## Summary

This notebook demonstrated how to use Rockfish's Entity Data Generator to create synthetic Cisco Netflow data. Key features:

### Realistic Protocol-Port Patterns
- **TCP (protocol 6)**: HTTP (80), HTTPS (443), SSH (22), SMTP (25), FTP (21), MySQL (3306), RDP (3389), etc.
- **UDP (protocol 17)**: DNS (53), NTP (123), SNMP (161)
- Destination ports derived from service type ensures consistency

### Port Number Ranges
- **Source ports**: Ephemeral range (49152-65535) as used by clients
- **Destination ports**: Well-known (0-1023) and registered (1024-49151) ports

### IPv4 Address Generation
- **Internal hosts**: Private 10.x.x.x range (Class A private network)
- **External hosts**: Public IP ranges (avoiding reserved/private ranges)
- Addresses generated as separate octets then combined

### Traffic Patterns
- Exponential distribution for bytes/packets (most flows are small)
- Realistic TCP flag combinations
- Type of Service (ToS) values for QoS
- Interface-based routing information

### Potential Extensions
- Add bidirectional flow matching
- Include ASN (Autonomous System Number) information
- Add geolocation data for external hosts
- Implement time-based traffic patterns (business hours vs. off-hours)
- Add ICMP flows for ping/traceroute traffic